In [2]:
from pathlib import Path
import numpy as np
import torch
from scipy.spatial.distance import cdist
import re


In [3]:
root = Path("/home/moritz.burmester/riem-score-metrics/experiments_urc/geodesics_out")
score_dir = root / "t005"
rbf_land_dir = root / "trbf_land"

real_lat_path = "/home/moritz.burmester/riem-score-metrics/experiments_urc/latents.npy"
stats_path = "/home/moritz.burmester/riem-score-metrics/experiments_urc/latents_stats.npz"

n_angles = 180
deg_per_idx = 2.0

real = np.load(real_lat_path).astype(np.float32)
stats = np.load(stats_path)

In [4]:
def gamma_star(real, letters, a0, steps, N):
    us = np.linspace(0, 1, N)
    idx = np.round(a0[:, None] + steps[:, None] * us[None, :]).astype(int) % n_angles  
    return real[letters[:, None] * n_angles + idx]                                      

def d_rmse(paths, real):
    # mean distance from each geodesic point to nearest real latent, RMSE over points
    P, N, D = paths.shape
    nn = cdist(paths.reshape(-1, D), real).min(1).reshape(P, N)  
    return np.sqrt((nn ** 2).mean(1)), nn                        

def gamma_rmse(paths, gstar):
    g = np.linalg.norm(paths - gstar, axis=-1)  
    return np.sqrt((g ** 2).mean(1)), g

def agg(arr):
    m = float(arr.mean())
    s = float(arr.std(ddof=1))
    return m, s

In [5]:

lams = [0.0, 0.1, 0.25, 0.5, 0.75, 1.0]
init = "slerp"
rows = []

configs = []
for lam in lams:
    tag = f"{init}_lam{lam}".replace(".", "")
    configs.append((f"lam={lam}", score_dir / f"geo_{tag}.pt"))

for name, path in configs:
    if not path.exists():
        print("missing:", path.name); continue
    g = torch.load(path, map_location="cpu")
    paths = g["paths"].numpy()
    letters, a0, steps = g["letters"], g["a0"], g["steps"]
    gstar = gamma_star(real, letters, a0, steps, paths.shape[1])

    d_per, _ = d_rmse(paths[:, 1:-1], real)     
    gstar_int = gstar[:, 1:-1]
    gm_per = np.sqrt((np.linalg.norm(paths[:, 1:-1] - gstar_int, axis=-1) ** 2).mean(1))

    dm, de = agg(d_per); gmm, gme = agg(gm_per)
    rows.append((name, dm, de, gmm, gme))
    print(f"{name:<10}  D-RMSE={dm:.3f} ± {de:.3f}   gamma*-RMSE={gmm:.3f} ± {gme:.3f}")

lam=0.0     D-RMSE=0.717 ± 0.637   gamma*-RMSE=0.981 ± 1.137
lam=0.1     D-RMSE=0.713 ± 0.634   gamma*-RMSE=0.947 ± 1.114
lam=0.25    D-RMSE=0.724 ± 0.662   gamma*-RMSE=0.938 ± 1.114
lam=0.5     D-RMSE=0.755 ± 0.711   gamma*-RMSE=0.961 ± 1.122
lam=0.75    D-RMSE=0.831 ± 0.787   gamma*-RMSE=1.058 ± 1.144
lam=1.0     D-RMSE=1.122 ± 1.018   gamma*-RMSE=1.439 ± 1.307


In [6]:
for name, fname in [("land", "geo_LAND.pt"), ("rbf", "geo_RBF.pt")]:
    path = rbf_land_dir / fname
    if not path.exists():
        print("missing:", path.name); continue
    g = torch.load(path, map_location="cpu")
    paths = g["paths"].numpy() if torch.is_tensor(g["paths"]) else np.asarray(g["paths"])
    if not all(k in g for k in ("letters", "a0", "steps")):
        print(f"{name}: no endpoint meta in file -> gamma* not comparable; D-RMSE only")
        d_per, _ = d_rmse(paths[:, 1:-1], real)
        dm, de = agg(d_per)
        print(f"{name:<10}  D-RMSE={dm:.3f} ± {de:.3f}   gamma*-RMSE=N/A")
        continue
    letters, a0, steps = g["letters"], g["a0"], g["steps"]
    gstar = gamma_star(real, letters, a0, steps, paths.shape[1])[:, 1:-1]
    d_per, _ = d_rmse(paths[:, 1:-1], real)
    gm_per = np.sqrt((np.linalg.norm(paths[:, 1:-1] - gstar, axis=-1) ** 2).mean(1))
    dm, de = agg(d_per); gmm, gme = agg(gm_per)
    print(f"{name:<10}  D-RMSE={dm:.3f} ± {de:.3f}   gamma*-RMSE={gmm:.3f} ± {gme:.3f}")

land        D-RMSE=0.700 ± 0.828   gamma*-RMSE=1.109 ± 1.479
rbf         D-RMSE=1.450 ± 1.700   gamma*-RMSE=2.502 ± 2.257


In [7]:

lams = [0.0, 0.1, 0.25, 0.5, 0.75, 1.0]
init = "lerp"
rows = []

configs = []
for lam in lams:
    tag = f"{init}_lam{lam}".replace(".", "")
    configs.append((f"lam={lam}", score_dir / f"geo_{tag}.pt"))

for name, path in configs:
    if not path.exists():
        print("missing:", path.name); continue
    g = torch.load(path, map_location="cpu")
    paths = g["paths"].numpy()
    letters, a0, steps = g["letters"], g["a0"], g["steps"]
    gstar = gamma_star(real, letters, a0, steps, paths.shape[1])

    d_per, _ = d_rmse(paths[:, 1:-1], real)     
    gstar_int = gstar[:, 1:-1]
    gm_per = np.sqrt((np.linalg.norm(paths[:, 1:-1] - gstar_int, axis=-1) ** 2).mean(1))

    dm, de = agg(d_per); gmm, gme = agg(gm_per)
    rows.append((name, dm, de, gmm, gme))
    print(f"{name:<10}  D-RMSE={dm:.3f} ± {de:.3f}   gamma*-RMSE={gmm:.3f} ± {gme:.3f}")

lam=0.0     D-RMSE=1.942 ± 1.786   gamma*-RMSE=2.318 ± 2.159
lam=0.1     D-RMSE=1.728 ± 1.606   gamma*-RMSE=2.110 ± 2.041
lam=0.25    D-RMSE=1.633 ± 1.538   gamma*-RMSE=2.049 ± 2.048
lam=0.5     D-RMSE=1.600 ± 1.538   gamma*-RMSE=2.076 ± 2.121
lam=0.75    D-RMSE=1.662 ± 1.562   gamma*-RMSE=2.228 ± 2.169
lam=1.0     D-RMSE=2.063 ± 1.621   gamma*-RMSE=2.874 ± 2.135


In [10]:
inits = ["lerp", "slerp"]
rows = []

configs = []
for init in inits:
    tag = f"raw_{init}".replace(".", "")
    configs.append((f"init={init}", score_dir / f"geo_{tag}.pt"))

for name, path in configs:
    if not path.exists():
        print("missing:", path.name); continue
    g = torch.load(path, map_location="cpu")
    paths = g["paths"].numpy()
    letters, a0, steps = g["letters"], g["a0"], g["steps"]
    gstar = gamma_star(real, letters, a0, steps, paths.shape[1])

    d_per, _ = d_rmse(paths[:, 1:-1], real)     
    gstar_int = gstar[:, 1:-1]
    gm_per = np.sqrt((np.linalg.norm(paths[:, 1:-1] - gstar_int, axis=-1) ** 2).mean(1))

    dm, de = agg(d_per); gmm, gme = agg(gm_per)
    rows.append((name, dm, de, gmm, gme))
    print(f"{name:<10}  D-RMSE={dm:.3f} ± {de:.3f}   gamma*-RMSE={gmm:.3f} ± {gme:.3f}")

init=lerp   D-RMSE=2.082 ± 1.490   gamma*-RMSE=2.397 ± 1.826
init=slerp  D-RMSE=1.302 ± 1.442   gamma*-RMSE=1.461 ± 1.730
